# 4 · Refining the graph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vitormeriat/text4gcn/blob/main/docs/tutorials/04_refining_the_graph.ipynb)

Tutorial 3 showed that a graph with far fewer edges can classify almost as
well. Choosing a builder is one way to get there. **Graph refinement** is
another: it takes a graph that is already built and removes rare words and
weak edges before training, without rebuilding anything.

We first **measure** what each refinement removes, which takes seconds and
needs no training. Then we **train** on a few of the refined graphs.

> **Runtime:** about 5 min on a Colab T4 GPU and about 20 min on CPU.

| # | Tutorial | Question it answers |
|---|---|---|
| 1 | Text as a graph | What does the graph look like, and why does it help? |
| 2 | Your first classifier | How do I go from raw text to test accuracy? |
| 3 | Choosing a graph builder | How should words be connected? |
| 4 | **Refining the graph** | Can a smaller graph do as well? |
| 5 | Bring your own corpus | How do I use my own (non-English) data? |

In [ ]:
%pip install -q text4gcn
# Release candidate from TestPyPI (dependencies still come from PyPI):
# %pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "text4gcn==2.0.0rc2"

In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from text4gcn import GNN, Builder, RefinementConfig, TextPipeline, apply_refinement
from text4gcn.builder import Frequency
from text4gcn.datasets import twenty_newsgroups
from text4gcn.models.utils import LoadData

In [ ]:
NAME = "20NG4"
DATA_DIR = Path("data")
LOG_DIR = Path("logs")
SEED = 1234
CATEGORIES = ["alt.atheism", "soc.religion.christian", "comp.graphics", "sci.med"]

In [ ]:
shutil.rmtree(DATA_DIR, ignore_errors=True)   # TextPipeline never overwrites its outputs
twenty_newsgroups(DATA_DIR, name=NAME, categories=CATEGORIES)
TextPipeline(dataset_name=NAME, dataset_path=DATA_DIR, seed=SEED).execute(validation_ratio=0.10)
Frequency(dataset_name=NAME, dataset_path=DATA_DIR).tfidf(window_size=20)

## The knobs

`RefinementConfig` combines node filters and edge pruning. Document nodes are
always kept (`keep_docs=True`), so the train, validation and test sets do not
change.

| Option | Removes |
|---|---|
| `min_doc_freq=k` | words that occur in fewer than *k* documents |
| `max_doc_freq_ratio=r` | words that occur in more than a fraction *r* of the documents |
| `edge_weight_threshold=w` | edges with a PMI or TF-IDF weight below *w* |
| `top_k_edges_per_node=k` | all but the *k* heaviest edges of each node |

See the [refinement guide](https://vitormeriat.github.io/text4gcn/guides/refinement/) for the full list.

## Measure before you train

`LoadData(...).load_corpus()` loads the graph and features that `GNN` would
train on. `apply_refinement(..., return_metadata=True)` returns the refined
graph together with statistics, without training.

In [ ]:
predata = LoadData(path=DATA_DIR, dataset=NAME, builder=Builder.Tfidf).load_corpus()
weights = predata[0].tocsr().data          # PMI and TF-IDF weights of every edge

quantiles = pd.Series(np.quantile(weights, [0.1, 0.25, 0.5, 0.75, 0.9]),
                      index=["10%", "25%", "50%", "75%", "90%"])
print(f"{len(weights):,} edges; weight quantiles:")
quantiles.round(2)

Most weights are well above 1, so a threshold like `0.05` would remove
almost nothing. **Pick thresholds from the distribution of your own graph.**
Here we use the median (removes half of the edges) and the 90th percentile
(removes 90%).

In [ ]:
median, p90 = quantiles["50%"], quantiles["90%"]
candidates = {
    "original": RefinementConfig(),
    "min_doc_freq=3": RefinementConfig(min_doc_freq=3),
    "min_doc_freq=10": RefinementConfig(min_doc_freq=10),
    f"weight ≥ {median:.2f} (median)": RefinementConfig(edge_weight_threshold=median),
    f"weight ≥ {p90:.2f} (p90)": RefinementConfig(edge_weight_threshold=p90),
    "top-20 edges/node": RefinementConfig(top_k_edges_per_node=20),
}

sizes = pd.DataFrame({
    name: apply_refinement(predata, config, dataset_path=DATA_DIR, dataset_name=NAME,
                           return_metadata=True)[1]["stats"]
    for name, config in candidates.items()
}).T
sizes["edges kept"] = sizes.edges_after / sizes.loc["original", "edges_after"]
sizes[["nodes_refined", "edges_after", "edges kept"]].style.format(
    {"nodes_refined": "{:,}", "edges_after": "{:,}", "edges kept": "{:.0%}"})

The filters act differently:

* `min_doc_freq` removes **word nodes**: rare words and all their edges.
* Weight thresholds and `top_k_edges_per_node` keep every node and remove
  **edges**. `top-k` is the most aggressive: hub words lose most of their links.

## Does accuracy survive?

We train on the original graph and on three refined ones. `GNN` applies the
refinement after loading the graph and prints the same statistics as above.

In [ ]:
to_train = ["original", "min_doc_freq=3", f"weight ≥ {median:.2f} (median)", "top-20 edges/node"]

accuracy = {}
for name in to_train:
    result = GNN(dataset_name=NAME, dataset_path=DATA_DIR, log_dir=LOG_DIR, builder=Builder.Tfidf,
                 seed=SEED, refinement_config=candidates[name]).fit()
    accuracy[name] = result.test_accuracy

summary = sizes.loc[to_train, ["nodes_refined", "edges kept"]].assign(test_accuracy=pd.Series(accuracy))
summary.style.format({"nodes_refined": "{:,}", "edges kept": "{:.0%}", "test_accuracy": "{:.2%}"})

In [ ]:
ax = summary.plot.scatter(x="edges kept", y="test_accuracy", s=80, figsize=(7, 4),
                          title="Accuracy vs. fraction of edges kept")
for k, (name, row) in enumerate(summary.iterrows()):
    right_side = row["edges kept"] > 0.9            # label to the left of points near 100%
    ax.annotate(name, (row["edges kept"], row.test_accuracy), textcoords="offset points",
                xytext=(-8 if right_side else 8, 8 if k % 2 else -14),
                ha="right" if right_side else "left")
ax.set_ylabel("test accuracy")
ax.margins(x=0.08, y=0.2)
ax.xaxis.set_major_formatter("{x:.0%}")
ax.yaxis.set_major_formatter("{x:.1%}");

## What the numbers say

On our CPU run (seed 1234):

* `min_doc_freq=3` removed about 600 rare words and **lost nothing** (86.9% vs
  86.7%). Fewer nodes means less memory, because the dense GCN grows with the
  square of the node count.
* Dropping the weaker half of the edges (weight ≥ median) cost only about
  **0.2 points** (86.5%). Moderate pruning is safe.
* `top-20 edges/node` kept only **8%** of the edges, but accuracy dropped by
  about **4.5 points** (82.3%).
* Compare that with tutorial 3: the **Word2Vec RWF** graph kept about 6% of the
  edges and lost only about **1 point**.

Removing edges by *weight* loses useful structure. Choosing which words to
connect by *meaning*, as the RWF does, keeps it. That is the central argument
of Pereira & de Castro (DCAI 2022).

## Takeaways

* **Measure first.** Graph statistics take seconds; training takes minutes.
* **Prune words, not just edges.** `min_doc_freq` is a safe first step.
* For an order-of-magnitude smaller graph, **change the builder** (tutorial 3)
  rather than pruning TF-IDF + PMI.
* Refinement works with any builder: pass `refinement_config` together with `builder`.

**Next:** so far we have used a benchmark. [5 · Bring your own corpus](https://vitormeriat.github.io/text4gcn/tutorials/05_bring_your_own_corpus/)
shows how to prepare your own data, in any language.